# AVSpeech Dataset Downloader (Colab)

Downloads AVSpeech clips from YouTube directly to Google Drive.

**Instructions:**
1. Run cells in order
2. Authorize Google Drive access when prompted
3. Let it run — progress is saved, so you can resume if the session disconnects

**Target:** ~25K clips for contrastive pretraining

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q yt-dlp huggingface_hub

In [ ]:
import csv
import json
import os
import random
import subprocess
import time
from pathlib import Path
from IPython.display import clear_output

# ============================================================
# CONFIGURATION - Edit these as needed
# ============================================================
TARGET_CLIPS = 25000        # Total clips we want
ATTEMPT_MULTIPLIER = 2.2    # Attempt 2.2x more to account for ~48% unavailable rate
SEED = 42                   # Same seed as local script for consistency
DELAY = 1.5                 # Seconds between downloads (Colab has fresh IP, can be lower)
DOWNLOAD_TIMEOUT = 45       # Seconds before giving up on a single clip

# Google Drive paths
DRIVE_BASE = Path('/content/drive/MyDrive/5330 - CVPR/SyncGuard')
DRIVE_DATA = DRIVE_BASE / 'data' / 'raw' / 'AVSpeech'
CLIPS_DIR = DRIVE_DATA / 'clips'
LOG_PATH = DRIVE_DATA / 'download_log.json'
MANIFEST_PATH = DRIVE_DATA / 'manifest.csv'
CSV_PATH = DRIVE_DATA / 'avspeech_train.csv'

# Create directories
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')
print(f'Clips dir:  {CLIPS_DIR}')
print(f'Target:     {TARGET_CLIPS} clips')

## 2. Download AVSpeech Metadata CSV

In [ ]:
if not CSV_PATH.exists():
    print('Downloading AVSpeech train CSV from Hugging Face...')
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id='bbrothers/avspeech-metadata',
        filename='avspeech_train.csv',
        repo_type='dataset',
        local_dir=str(DRIVE_DATA),
    )
    print(f'Saved to {CSV_PATH}')
else:
    print(f'CSV already exists: {CSV_PATH}')

print(f'Size: {CSV_PATH.stat().st_size / 1024 / 1024:.1f} MB')

## 3. Load and Select Clips

In [ ]:
def load_csv(csv_path):
    rows = []
    with open(csv_path) as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 5:
                continue
            rows.append({
                'youtube_id': row[0].strip(),
                'start': float(row[1].strip()),
                'end': float(row[2].strip()),
                'face_x': float(row[3].strip()),
                'face_y': float(row[4].strip()),
            })
    return rows

print('Loading CSV...')
all_rows = load_csv(CSV_PATH)
print(f'Total entries: {len(all_rows):,}')

# Shuffle deterministically and select enough to hit our target
random.seed(SEED)
random.shuffle(all_rows)
num_to_attempt = int(TARGET_CLIPS * ATTEMPT_MULTIPLIER)
selected = all_rows[:num_to_attempt]
print(f'Will attempt: {len(selected):,} clips (to get ~{TARGET_CLIPS:,} successful)')

## 4. Download Functions

In [ ]:
def load_log():
    if LOG_PATH.exists():
        with open(LOG_PATH) as f:
            log = json.load(f)
        if isinstance(log.get('failed'), list):
            log['failed'] = {yt_id: 'unknown' for yt_id in log['failed']}
        return log
    return {'successful': [], 'failed': {}, 'unavailable': []}


def save_log(log):
    with open(LOG_PATH, 'w') as f:
        json.dump(log, f)


def download_clip(youtube_id, start, end, output_path):
    url = f'https://www.youtube.com/watch?v={youtube_id}'
    cmd = [
        'yt-dlp',
        '--quiet', '--no-warnings',
        '--download-sections', f'*{start}-{end}',
        '-f', 'bestvideo[height<=360][ext=mp4]+bestaudio[ext=m4a]/best[height<=360][ext=mp4]/best[height<=360]',
        '--merge-output-format', 'mp4',
        '-o', str(output_path),
        '--socket-timeout', '10',
        '--retries', '1',
        '--no-playlist',
        url,
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=DOWNLOAD_TIMEOUT)
        if result.returncode == 0 and output_path.exists():
            return True, 'ok'
        stderr = result.stderr.strip()
        if 'Video unavailable' in stderr or 'Private video' in stderr:
            return False, 'unavailable'
        if 'Sign in to confirm' in stderr:
            return False, 'bot_check'
        return False, stderr[:150] if stderr else 'unknown'
    except subprocess.TimeoutExpired:
        # Clean up partial file
        for f in output_path.parent.glob(f'{output_path.stem}*'):
            f.unlink(missing_ok=True)
        return False, 'timeout'
    except Exception as e:
        return False, str(e)[:150]


def write_manifest(entries):
    with open(MANIFEST_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['youtube_id', 'start', 'end', 'face_x', 'face_y', 'clip_path'])
        for e in entries:
            writer.writerow([
                e['youtube_id'], e['start'], e['end'],
                e['face_x'], e['face_y'], e['clip_path'],
            ])

print('Download functions ready.')

## 5. Download Clips

This cell is **resumable** — if the session disconnects, just re-run from here.

It will automatically:
- Skip already-downloaded clips
- Skip previously unavailable clips
- Stop once we hit the target count
- Save progress every 25 clips

In [ ]:
# Load existing progress
log = load_log()
already_done = set(log['successful'] + log['unavailable'] + list(log['failed'].keys()))

success_count = len(log['successful'])
unavail_count = len(log['unavailable'])
fail_count = len(log['failed'])

print(f'Existing progress: {success_count} OK, {unavail_count} unavailable, {fail_count} failed')
print(f'Target: {TARGET_CLIPS} clips')

if success_count >= TARGET_CLIPS:
    print(f'Already have {success_count} clips — target reached!')
else:
    # Rebuild manifest from existing successful downloads
    manifest_entries = []
    successful_set = set(log['successful'])
    for row in selected:
        if row['youtube_id'] in successful_set:
            clip_path = CLIPS_DIR / f"{row['youtube_id']}.mp4"
            if clip_path.exists():
                manifest_entries.append({**row, 'clip_path': str(clip_path)})

    start_time = time.time()
    consecutive_bot_checks = 0

    for i, row in enumerate(selected):
        # Stop if we've reached target
        if success_count >= TARGET_CLIPS:
            print(f'\nTarget of {TARGET_CLIPS} clips reached!')
            break

        yt_id = row['youtube_id']
        if yt_id in already_done:
            continue

        clip_path = CLIPS_DIR / f'{yt_id}.mp4'

        # File exists on disk but not in log
        if clip_path.exists():
            log['successful'].append(yt_id)
            already_done.add(yt_id)
            manifest_entries.append({**row, 'clip_path': str(clip_path)})
            success_count += 1
            continue

        ok, msg = download_clip(yt_id, row['start'], row['end'], clip_path)

        if ok:
            log['successful'].append(yt_id)
            manifest_entries.append({**row, 'clip_path': str(clip_path)})
            success_count += 1
            consecutive_bot_checks = 0
        elif msg == 'unavailable':
            log['unavailable'].append(yt_id)
            unavail_count += 1
            consecutive_bot_checks = 0
        elif msg == 'bot_check':
            log['failed'][yt_id] = msg
            fail_count += 1
            consecutive_bot_checks += 1
            if consecutive_bot_checks >= 5:
                print(f'\n!! YouTube bot detection triggered. Pausing 5 minutes...')
                save_log(log)
                time.sleep(300)
                consecutive_bot_checks = 0
        else:
            log['failed'][yt_id] = msg
            fail_count += 1
            consecutive_bot_checks = 0

        already_done.add(yt_id)
        time.sleep(DELAY)

        # Progress update every 25 clips
        total = success_count + unavail_count + fail_count
        if total % 25 == 0:
            elapsed = time.time() - start_time
            rate = (total - (len(log['successful']) - success_count + unavail_count + fail_count)) / elapsed if elapsed > 0 else 0
            remaining = TARGET_CLIPS - success_count
            clear_output(wait=True)
            print(f'Progress: {success_count:,}/{TARGET_CLIPS:,} clips downloaded')
            print(f'Attempted: {total:,} | Unavailable: {unavail_count:,} | Failed: {fail_count:,}')
            print(f'Elapsed: {elapsed/60:.1f} min | ~{remaining} clips remaining')
            save_log(log)

    # Final save
    save_log(log)
    write_manifest(manifest_entries)

    elapsed = time.time() - start_time
    total_size = sum(f.stat().st_size for f in CLIPS_DIR.glob('*.mp4'))
    print(f'\n{"="*60}')
    print(f'Download complete in {elapsed/60:.1f} minutes')
    print(f'  Successful:   {success_count:,}')
    print(f'  Unavailable:  {unavail_count:,}')
    print(f'  Failed:       {fail_count:,}')
    print(f'  Total storage: {total_size / 1024**3:.2f} GB')
    print(f'  Manifest:     {MANIFEST_PATH}')
    print(f'{"="*60}')

## 6. Verify Downloads

In [ ]:
# Quick stats
clip_files = list(CLIPS_DIR.glob('*.mp4'))
total_size = sum(f.stat().st_size for f in clip_files)

print(f'Total clips on Drive: {len(clip_files):,}')
print(f'Total storage:        {total_size / 1024**3:.2f} GB')
print(f'Avg clip size:        {total_size / len(clip_files) / 1024:.0f} KB' if clip_files else 'N/A')
print(f'Manifest:             {MANIFEST_PATH}')

# Verify manifest matches files
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest_count = sum(1 for _ in csv.reader(f)) - 1  # minus header
    print(f'Manifest entries:     {manifest_count:,}')

## 7. Copy Local Clips to Drive (Optional)

If you already downloaded clips locally and uploaded them to Drive separately,
run this cell to merge them into the same directory and update the manifest.

In [ ]:
# If you uploaded local clips to a separate folder, set the path here:
# LOCAL_UPLOAD_DIR = Path('/content/drive/MyDrive/SyncGuard/data/raw/AVSpeech/local_clips')
#
# if LOCAL_UPLOAD_DIR.exists():
#     import shutil
#     for f in LOCAL_UPLOAD_DIR.glob('*.mp4'):
#         dest = CLIPS_DIR / f.name
#         if not dest.exists():
#             shutil.move(str(f), str(dest))
#     print(f'Merged local clips. Total: {len(list(CLIPS_DIR.glob("*.mp4"))):,}')